In [ ]:
%load_ext autoreload
%autoreload 2

### Update with yeast display data

 - Contains YD updating and downstream analyses.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PepCL_Manuscript":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find PepCL_Manuscript project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

PROJECT_ROOT: /home/pchati/PepCL_Manuscript
DATA_DIR: /home/pchati/PepCL_Manuscript/Data
PROJECT_ROOT in sys.path: True


In [ ]:
# imports
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.mhcprime_inference import *
from src.plotting import *

# misc
import warnings
warnings.filterwarnings("ignore")
device="cuda"

In [ ]:
# change gpu
GPU_ID = 0
# GPU_ID = 1
# GPU_ID = 2
# GPU_ID = 3

torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")

Using GPU: NVIDIA H100 80GB HBM3 (ID: 0)


### Get datasets

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
from Data.mhc_pseudo_seqs import mhc_seq_dict
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### MS/YD train and test sets

In [ ]:
# ms train and test sets
datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data.csv", index_col=0)
ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)

In [ ]:
# yd train and test sets
holec_datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "Holec_et_al_2025"
holec_train_set = pd.read_csv(f"{holec_datasets_dir}/holec_train_set.csv", index_col=0)
holec_test_set  = pd.read_csv(f"{holec_datasets_dir}/holec_test_set.csv", index_col=0)

In [ ]:
# get the pathogen sets
pathogen_set_datasets_dir= DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets" / "Pathogen_Sets"
ms_cov_set=pd.read_csv(f"{pathogen_set_datasets_dir}/ms_cov_set.csv", index_col=0)
ms_zika_set=pd.read_csv(f"{pathogen_set_datasets_dir}/ms_zika_set.csv", index_col=0)
ms_mtb_set=pd.read_csv(f"{pathogen_set_datasets_dir}/ms_mtb_set.csv", index_col=0)

In [ ]:
# merge to create ms/holec train sets

# assign domain
ms_train_data["domain_id"]=0
holec_train_set["domain_id"]=1

ms_holec_train_set=pd.concat([ms_train_data, holec_train_set])

# reset index
ms_holec_train_set.reset_index(inplace=True, drop=True)

# print stats
print_data_stats(ms_holec_train_set)

Total: 39,380,571
Pos: 771,756 | 1.96%
Neg: 38,608,815 | 98.04%
N alleles: 192


In [ ]:
# match allele and length distribution between domains in train
ms_holec_train_set["seq_len"]=ms_holec_train_set.seq.apply(len) # seq.str.replace("$", "") check for padding

ms_holec_train_set_matched_allele_length=ms_holec_train_set[ms_holec_train_set.allele.isin(holec_train_set.allele.unique().tolist())].query("seq_len == 9")

# reset index
ms_holec_train_set_matched_allele_length.reset_index(inplace=True, drop=True)

# print stats
print_data_stats(ms_holec_train_set_matched_allele_length)

Total: 6,384,285
Pos: 124,770 | 1.95%
Neg: 6,259,515 | 98.05%
N alleles: 16


In [ ]:
# full processing for running
ms_holec_train_set_matched_allele_length=full_process_data(ms_holec_train_set_matched_allele_length, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_holec_train_set_matched_allele_length=create_seq_allele_column(ms_holec_train_set_matched_allele_length)

holec_test_set=full_process_data(holec_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
holec_test_set=create_seq_allele_column(holec_test_set)

ms_test_data=full_process_data(ms_test_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_test_data=create_seq_allele_column(ms_test_data)

ms_cov_set=full_process_data(ms_cov_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=True)
ms_cov_set=create_seq_allele_column(ms_cov_set)

ms_zika_set=full_process_data(ms_zika_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=True)
ms_zika_set=create_seq_allele_column(ms_zika_set)

ms_mtb_set=full_process_data(ms_mtb_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=True)
ms_mtb_set=create_seq_allele_column(ms_mtb_set)

In [ ]:
# rename donor cols
ms_cov_set.rename(columns={"cell_type":"group"}, inplace=True)
ms_zika_set.rename(columns={"cell_line":"group"}, inplace=True)
ms_mtb_set.rename(columns={"donor":"group"}, inplace=True)

#### Background datasets

In [ ]:
%%time
# for ranking
chp_background_10k_per_allele_x_len_processed=pd.read_csv(f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.csv")

CPU times: user 27.8 s, sys: 5.36 s, total: 33.2 s
Wall time: 33.2 s


In [ ]:
# for y84a analyses
swissprot_proteome_background_for_holec_df_processed=pd.read_csv(f"{DATA_DIR}/swissprot_proteome_background_for_holec_df_processed.csv", index_col=0)

### Load MHCPrime

In [ ]:
# setup tokenizer
tokenizer = AminoAcidTokenizer()

In [ ]:
# set base model params
base_model_params={
        'vocab_size': tokenizer.vocab_size,
        'embed_dim': 480,
        'fc_hidden_dims': [240, 120], 
        'fc_dropout': 0.1,
        'n_heads': 16,
        'max_peptide_len': 34,
        'max_mhc_len': 34,
        "n_peptide_layers": 2,
        "n_mhc_layers": 1,
        'n_concat_layers': 3,
        'add_pos_enc': True,
        'use_segment_embedder': True,
        'dim_feedforward_multiplier': 1,
        'debugging': False,
        'fc_output_dim': 1,
        'pooling': 'cls', # cls, mean, min, max, attn, for sequence ffnn head
        "ffnn_use_layernorm": False, 
        'use_feature_table': True,
        "feature_table": processed_feature_table,
        "sequence_embedder_feature_names": feature_names,
        'sequence_embedder_feature_mode': "concat",  # replace or concat
        'sequence_embedder_tokenizer': tokenizer,
        "peptide_start_pos": 1, 
        "mhc_start_pos": 35, 
        "pos_enc_type": "sincos", 
        "seed": 42,

        # peptide conformer
        "add_peptide_conformer": True, 
        "peptide_conformer_num_layers": 2,
        "peptide_conformer_n_heads": 16,
        "peptide_conformer_conv_kernel": 5,
        "peptide_conformer_dropout": 0.0,

        # mhc conformer
        "add_mhc_conformer": False, 
        "mhc_conformer_num_layers": 2,
        "mhc_conformer_n_heads": 16,
        "mhc_conformer_conv_kernel": 5,
        "mhc_conformer_dropout": 0.0,

        # fusion conformer parameters
        "add_fusion_conformer": True, 
        "fusion_conformer_num_layers": 1,
        "fusion_conformer_n_heads": 16,
        "fusion_conformer_conv_kernel": 7,
        "fusion_conformer_dropout": 0.0,
        "fusion_conformer_mask_bos": True,
        "ff_pre_post_swiglu": False,

        # legacy init
        "legacy_init_compat": True,
}

In [ ]:
%%time
# load base model (optionally, can load model_params directly)
run_name = "MHCPrime_Base"
run_dir = MODEL_CHECKPOINTS_DIR / run_name
model_dir = run_dir / "Checkpoints"

mhcprime=init_model(base_model_params, print_check=False, print_params=True)
mhcprime.load_state_dict(torch.load(f"{model_dir}/model_final.pt"))
mhcprime.eval()
print("Model loaded from checkpoint...")
clear_all_gpu_memory()

[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
All GPU memory has been cleared.
Current GPU memory allocated: 0.10 GB
Current GPU memory reserved: 0.12 GB
CPU times: user 1.01 s, sys: 711 ms, total: 1.72 s
Wall time: 1.73 s


### PepCL updating

#### Run PepCL

In [ ]:
pepcl_base_params=dict(
            tokenizer=tokenizer,
            device="cuda",

            # steps
            steps = 200,
            
            output_dir = None,
            save_every_n_steps = 2000,
            seed = 42,
            num_workers = 16,
            collate_fn = collate_fn,

            # sampling
            num_ms_pos = 2000,
            num_nonms_pos = 2000,
            ms_neg_ratio = 1.0,
            nonms_neg_ratio = 1.0,

            auto_calibrate = True,
            calibration_steps = 1,

            ms_batch_size = 3072,
            nonms_batch_size = 3072,

            # optimization
            encoder_lr = 2e-4,
            decoder_lr = 2e-4,
            weight_decay = 0.01,
            warmup_ratio = 0.1,
            update_domain_embeddings = False,

            # continual learning parameters
            primary_task_id = 1,
            sequential_backprop = True,

            # losses
            ms_loss_type = "logsmoothap",
            ms_loss_alpha = 10.0,
            ms_lambda = 1.0,
            nonms_loss_type = "logsmoothap",
            nonms_loss_alpha = 10.0,
            ft_lambda = 1.0,

            # drift regularization
            dr_lambda = 1e-5,
            dr_loss_type = "mse"
)

In [ ]:
# init models
holec_ft = copy.deepcopy(mhcprime)
holec_pepcl = copy.deepcopy(mhcprime)
clear_all_gpu_memory()

All GPU memory has been cleared.
Current GPU memory allocated: 0.31 GB
Current GPU memory reserved: 0.35 GB


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=holec_ft,
            train_df=ms_holec_train_set_matched_allele_length,
            use_replay = False,
            **pepcl_base_params,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=16, A_eff=6.56, balance=0.410, target_pos/cycle=1536, q_pos median=74.0 [46, 344], reuse median=0.0198, reuse max=0.0608, neg_ratio=1.0
  FT: alleles=16, A_eff=12.41, balance=0.775, target_pos/cycle=1536, q_pos median=93.5 [80, 120], reuse median=0.0622, reuse max=0.1874, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [01:09<00:00,  2.86it/s, MS=0.000000, FT=0.011590, DR=0.000000]


CPU times: user 1min 16s, sys: 12.3 s, total: 1min 28s
Wall time: 1min 27s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=holec_pepcl,
            train_df=ms_holec_train_set_matched_allele_length,
            use_replay = True,
            **pepcl_base_params,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=16, A_eff=6.56, balance=0.410, target_pos/cycle=1536, q_pos median=74.0 [46, 344], reuse median=0.0198, reuse max=0.0608, neg_ratio=1.0
  FT: alleles=16, A_eff=12.41, balance=0.775, target_pos/cycle=1536, q_pos median=93.5 [80, 120], reuse median=0.0622, reuse max=0.1874, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [02:38<00:00,  1.26it/s, MS=0.010214, FT=0.015621, DR=0.006153]


CPU times: user 2min 41s, sys: 20.7 s, total: 3min 2s
Wall time: 2min 59s


In [ ]:
%%time
"""
Save models
"""

# save_models=True
save_models=False

models_to_save={
    "holec_ft": holec_ft,
    "holec_pepcl": holec_pepcl,
}

yd_model_checkpoints_dir = MODEL_CHECKPOINTS_DIR / "YD_Models"
os.makedirs(yd_model_checkpoints_dir, exist_ok=True)

if save_models:
    for model_name, model in models_to_save.items():
        print(model_name)
        torch.save(model.state_dict(), f"{yd_model_checkpoints_dir}/{model_name}.pt")

holec_ft
holec_pepcl
CPU times: user 230 ms, sys: 177 ms, total: 407 ms
Wall time: 403 ms


#### Inference

In [ ]:
# get models
base_model=copy.deepcopy(mhcprime)

base_and_ft_models={
    # base model
    "mhcprime": base_model,

    # updated models
    "holec_ft": holec_ft,
    "holec_pepcl": holec_pepcl,
}

In [ ]:
%%time
"""
Load models (optional if running inference in a different session)
"""

load_models=False
# load_models=True

if load_models:
    base_and_ft_models={"mhcprime": base_model}
    yd_model_checkpoints_dir = MODEL_CHECKPOINTS_DIR / "YD_Models"
    for model_name in os.listdir(yd_model_checkpoints_dir):
        print(model_name)
        if model_name.endswith(".pt"):
            model_name_cleaned=model_name.removesuffix(".pt")
            print(model_name_cleaned)
            model=init_model(base_model_params, print_check=False, print_params=True)
            model.load_state_dict(torch.load(f"{yd_model_checkpoints_dir}/{model_name}"))
            base_and_ft_models[model_name_cleaned]=model

In [ ]:
%%time
# build cache for background set
build_caches=True
if build_caches:
    data_to_run_path=f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.pt"
    build_token_cache(chp_background_10k_per_allele_x_len_processed, data_to_run_path, tokenizer, use_domain_id=False)

13774791it [05:19, 43140.28it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/chp_background_10k_per_allele_x_len_processed.pt  (3746.7 MB)
CPU times: user 5min 20s, sys: 9.35 s, total: 5min 30s
Wall time: 5min 28s


In [ ]:
%%time
"""
Score background
"""
# chp background
data_to_run_path=f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.pt"

# run_bg=False
run_bg=True

if run_bg:
    for model_name, model in base_and_ft_models.items():
        print(model_name)
        model.eval()
        with torch.no_grad():
            num_workers=16
            output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=num_workers, compile_model=False, use_domain_id=False)

        chp_background_10k_per_allele_x_len_processed[model_name]=output_scores.detach().cpu().tolist() 

mhcprime
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 4484/4484 [09:05<00:00,  8.22it/s]


holec_ft
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 4484/4484 [09:06<00:00,  8.21it/s]


holec_pepcl
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 4484/4484 [09:04<00:00,  8.23it/s]


CPU times: user 27min 40s, sys: 1min 7s, total: 28min 47s
Wall time: 27min 35s


In [ ]:
%%time
"""
Save ranking DF with scores
"""
# chp_background_10k_per_allele_x_len_processed.to_csv(f"{DATA_DIR}/chp_background_10k_per_allele_x_len_processed.csv")

CPU times: user 56.6 s, sys: 4.07 s, total: 1min
Wall time: 1min


In [ ]:
%%time
# build cache for the sp df
build_caches=True
if build_caches:
    data_to_run_path=f"{DATA_DIR}/swissprot_proteome_background_for_holec_df_processed.pt"
    build_token_cache(swissprot_proteome_background_for_holec_df_processed, data_to_run_path, tokenizer, use_domain_id=False)

1596336it [00:37, 43016.61it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/swissprot_proteome_background_for_holec_df_processed.pt  (434.2 MB)
CPU times: user 37.4 s, sys: 1.11 s, total: 38.5 s
Wall time: 38.3 s


In [ ]:
%%time
"""
Score sp background
"""
# sp background
data_to_run_path=f"{DATA_DIR}/swissprot_proteome_background_for_holec_df_processed.pt"

# run_bg=False
run_bg=True

if run_bg:
    for model_name, model in base_and_ft_models.items():
        print(model_name)
        model.eval()
        with torch.no_grad():
            num_workers=16
            output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=num_workers, compile_model=False, use_domain_id=False)

        swissprot_proteome_background_for_holec_df_processed[model_name]=output_scores.detach().cpu().tolist() 

mhcprime
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 520/520 [01:06<00:00,  7.80it/s]


holec_ft
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 520/520 [01:07<00:00,  7.75it/s]


holec_pepcl
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 520/520 [01:06<00:00,  7.79it/s]


CPU times: user 3min 12s, sys: 18.4 s, total: 3min 30s
Wall time: 3min 25s


In [ ]:
%%time
"""
Save SP background
"""
# swissprot_proteome_background_for_holec_df_processed.to_csv(f"{DATA_DIR}/swissprot_proteome_background_for_holec_df_processed.csv")

CPU times: user 5.68 s, sys: 404 ms, total: 6.08 s
Wall time: 6.08 s


In [ ]:
# add base model (if not already added)
# base_and_ft_models["mhcprime"]=base_model

In [ ]:
%%time
# ms_test_data
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ms_test_data, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ms_test_data[model_name]=output_scores.detach().cpu().tolist()

3214300it [01:12, 44159.64it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (874.3 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:10<00:00,  8.00it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:10<00:00,  8.03it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:10<00:00,  8.01it/s]


CPU times: user 7min 41s, sys: 26.5 s, total: 8min 7s
Wall time: 7min 52s


In [ ]:
%%time
# holec_test_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(holec_test_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    holec_test_set[model_name]=output_scores.detach().cpu().tolist()

160000it [00:03, 40706.79it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (43.5 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 53/53 [00:10<00:00,  5.15it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 53/53 [00:10<00:00,  5.12it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 53/53 [00:10<00:00,  4.99it/s]


CPU times: user 23.4 s, sys: 12.9 s, total: 36.3 s
Wall time: 37.8 s


In [ ]:
%%time
# ms_cov_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ms_cov_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ms_cov_set[model_name]=output_scores.detach().cpu().tolist()

186it [00:00, 19693.06it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (0.1 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:03<00:00,  3.95s/it]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:03<00:00,  3.97s/it]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:04<00:00,  4.84s/it]


CPU times: user 188 ms, sys: 12.8 s, total: 12.9 s
Wall time: 15.1 s


In [ ]:
%%time
# ms_zika_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ms_zika_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ms_zika_set[model_name]=output_scores.detach().cpu().tolist()

57it [00:00, 15270.52it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (0.0 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:04<00:00,  4.93s/it]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:05<00:00,  5.50s/it]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:05<00:00,  5.32s/it]


CPU times: user 240 ms, sys: 15.7 s, total: 16 s
Wall time: 18.1 s


In [ ]:
%%time
# ms_mtb_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ms_mtb_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ms_mtb_set[model_name]=output_scores.detach().cpu().tolist()

96it [00:00, 15005.90it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (0.0 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:05<00:00,  5.73s/it]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:05<00:00,  5.58s/it]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1/1 [00:05<00:00,  5.65s/it]


CPU times: user 234 ms, sys: 16.9 s, total: 17.1 s
Wall time: 19.1 s


In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
%%time
# get rank values
ms_cov_set["seq_len"]=ms_cov_set.seq.str.replace("$", "").apply(len)
ms_zika_set["seq_len"]=ms_zika_set.seq.str.replace("$", "").apply(len)
ms_mtb_set["seq_len"]=ms_mtb_set.seq.str.replace("$", "").apply(len)

score_cols=["mhcprime", "holec_ft", "holec_pepcl"]

ms_cov_set = add_background_percentile_ranks(
    test_df=ms_cov_set,
    background_df=chp_background_10k_per_allele_x_len_processed,
    allele_col="allele",
    seq_len_col="seq_len",
    score_cols=score_cols,
    rank_mode="global",
    suffix="_rank_r_chp",
    reverse=True,
    copy=True,
)

ms_zika_set = add_background_percentile_ranks(
    test_df=ms_zika_set,
    background_df=chp_background_10k_per_allele_x_len_processed,
    allele_col="allele",
    seq_len_col="seq_len",
    score_cols=score_cols,
    rank_mode="global",
    suffix="_rank_r_chp",
    reverse=True,
    copy=True,
)

ms_mtb_set = add_background_percentile_ranks(
    test_df=ms_mtb_set,
    background_df=chp_background_10k_per_allele_x_len_processed,
    allele_col="allele",
    seq_len_col="seq_len",
    score_cols=score_cols,
    rank_mode="global",
    suffix="_rank_r_chp",
    reverse=True,
    copy=True,
)


In [ ]:
%%time
"""
Save all sets back (note: they are now processed)
"""

save_dfs=False
# save_dfs=True

if save_dfs:
    # ms test set
    datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
    ms_test_data.to_csv(f"{datasets_dir}/ms_test_data.csv")

    # yd test set
    holec_datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "Holec_et_al_2025"
    holec_test_set.to_csv(f"{holec_datasets_dir}/holec_test_set.csv")

    # ms pathogen sets
    pathogen_set_datasets_dir= DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets" / "Pathogen_Sets"
    ms_cov_set.to_csv(f"{pathogen_set_datasets_dir}/ms_cov_set.csv")
    ms_zika_set.to_csv(f"{pathogen_set_datasets_dir}/ms_zika_set.csv")
    ms_mtb_set.to_csv(f"{pathogen_set_datasets_dir}/ms_mtb_set.csv")

CPU times: user 10.1 s, sys: 768 ms, total: 10.9 s
Wall time: 10.9 s
